In [ ]:
import sys
sys.stdout.flush()


In [ ]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from config import set_environment
from set_agents import get_agent
# for the keys - as explained early in chapter 2
set_environment()

# LangChain Common Expression Language (LCEL)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Create components
prompt = PromptTemplate.from_template("Tell me about {topic}")
llm = get_agent("openai")
output_parser = StrOutputParser()

# Chain them together using LCEL
chain = prompt | llm | output_parser

# Use the chain
result = chain.invoke({"topic": "AI Engineering"})
print(result)


# More complex expressions

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

from set_agents import get_agent


chat = get_agent("google")

# First chain generates a story
story_prompt = PromptTemplate.from_template("Write a short story about {topic}")
story_chain = story_prompt | chat | StrOutputParser()

# Second chain analyzes the story
analysis_prompt = PromptTemplate.from_template("Analyze the following story's mood:\n{story}")
analysis_chain = analysis_prompt | chat | StrOutputParser()

output_prompt = PromptTemplate.from_template(
    "Here's the story: \n{story}\n\nHere's the mood: \n{mood}"
)
# Combine chains
story_with_analysis = story_chain | analysis_chain 
# | output_prompt | chat | StrOutputParser()

# Run the combined chain
result = story_with_analysis.invoke({"topic": "soccer"})
print("\nAnalysis:\n", result)


In [ ]:
import langchain_core
dir(langchain_core.prompts)

In [ ]:
# prompt chaining:

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from set_agents import get_agent


llm = get_agent("openai")
# Create components
prompt = PromptTemplate.from_template(
    "Tell me about {topic}."
    )

# huggingface = get_agent("huggingface")
output_parser = StrOutputParser()

# Chain them together using LCEL
topic_chain = prompt | llm | output_parser


# Second chain analyzes the story
topic_analysis = PromptTemplate.from_template(
    """You are a technical career coach.

I am a senior ML Engineer at a Fintech company with ~8 years of experience,
strong Python skills, and experience with AWS and GCP.

Based on the input below, provide a detailed plan to transition into AI Engineering:

{story}
"""
)

topic_analysis_chain = topic_analysis | llm | StrOutputParser()

# topic_chain_with_analysis = (
#     topic_chain
#     | (lambda story: {"story": story})
#     | topic_analysis_chain
# )

enhanced_chain = (
    RunnablePassthrough.assign(
    story = topic_chain
).assign(
    analysis=topic_analysis_chain
    )
)
# Use the chain
result = enhanced_chain.invoke({"topic": "AI Engineering"})
# result = topic_chain_with_analysis.invoke({"topic": "AI Engineering"})
# story = topic_chain.invoke({"topic": "AI Engineering"})
# result = topic_analysis_chain.invoke({"story": story})
print("\nAI Engineering Tips: ", result)

In [ ]:
type(result)

In [ ]:
import json
from pathlib import Path

# Persist the dict result to JSON
if hasattr(result, "model_dump"):
    data = result.model_dump()
elif isinstance(result, dict):
    data = result
else:
    raise TypeError(f"result is type {type(result)}, not a dict")

output_json = Path("result.json")
output_json.write_text(json.dumps(data, indent=2))
print(f"Wrote dict result to {output_json.resolve()}")